# Glasses Detection Model


## Setup


### Imports


In [ ]:
from pathlib import Path
from sys import maxsize as sys_maxsize
from typing import Any, TypeAlias

import matplotlib.pyplot as plt
import numpy as np
import numpy.dtypes as npd
import numpy.typing as npt
from tensorflow_docs.plots import HistoryPlotter as tfdocs_HistoryPlotter


In [ ]:
import keras as k
import tensorflow as tf
from keras import optimizers as k_optimisers
from keras import regularizers as k_regularisers
from keras.layers import BatchNormalization as k_layers_BatchNormalisation
from keras.layers import Normalization as k_layers_Normalisation
from keras.layers import RandomColorJitter as k_layers_RandomColourJitter
from tensorflow import config as tf_config  # type: ignore
from tensorflow.data import AUTOTUNE as tf_AUTOTUNE  # type: ignore
from tensorflow.data import Dataset as tf_Dataset  # type: ignore


### Settings


In [ ]:
VERBOSITY = 2
VERBOSITY_AFFECTS_FIT = False

# Uses https://www.kaggle.com/datasets/mantasu/glasses-detector
DS_PATH = Path("../../data/datasets/glasses-detector/classification/anyglasses")
IGNORE_SUBDATASETS = ["cmu-face-images"]  # Poor dataset, incorrectly labelled

SEED = 125

##TEST_SPLIT = 0.1
##"""The ratio of data used for testing."""
##VAL_SPLIT = 0.3
##"""The ratio of training data split into validation data. Done after the test split."""

CACHE_PATH = Path("./temp/cache")
"""The stem needs to be a filename, not a folder."""
BUFFER_SIZE = 200
TRAIN_REPETITIONS = 0  # Setting to -1 enabled infinite looping.
TRAIN_REPETITIONS_IF_INFINITE = 50
# Both are pointless if data augmentation isn't enabled
VAL_REPEITIONS = 0
TEST_REPEITIONS = 0

IMG_LENGTH = 64
IMG_SIZE = (IMG_LENGTH, IMG_LENGTH)
IMG_SHAPE = IMG_SIZE + (3,)  # Tuple append
BATCH_SIZE = 200  # Try lowering for better VRAM performance

USE_PREPROCESSING = True
USE_AUGMENTATIONS = False
PREPROCESS_TRANSLATION_FACTOR = 0.1
PREPROCESS_ZOOM_FACTOR = (-0.2, 0)
PREPROCESS_ROTATION_FACTOR = 1.0
PREPROCESS_HUE_FACTOR = 1.0

MAX_EPOCHS = 300

INITIAL_LEARNING_RATE = 1e-3
DECAY_RATE = 0.2

STOPPING_PATIENCE = 10

REGULARISER_VAL = 1e-5
DROPOUT_VAL = 0.05

SAVE_PATH = Path("../panopticon/model_weights/glasses_detection.keras")


### Config


In [ ]:
GPUS: list[tf_config.PhysicalDevice] = tf_config.list_physical_devices(
	device_type="GPU"
)
if VERBOSITY > 0:
	display(f"{len(GPUS)} GPU(s): {GPUS}")


In [ ]:
try:
	tf_config.experimental.set_memory_growth(GPUS[0], True)
except Exception:
	print("Invalid device or cannot modify virtual devices once initialized.")
	pass

##tf_config.run_functions_eagerly(False)


In [ ]:
k.utils.set_random_seed(seed=SEED)


In [ ]:
np.set_printoptions(
	floatmode="fixed",  # Doesn't work??
	precision=2,
	threshold=sys_maxsize,
	linewidth=sys_maxsize,
	formatter={"float": lambda num: "{:.2f}".format(num)},
)


### Types


In [ ]:
type PathLabel = tuple[str, bool]

ImageDType: TypeAlias = np.uint8
LabelDType: TypeAlias = np.bool

type ImageBatch = npt.NDArray[ImageDType]
type LabelBatch = npt.NDArray[LabelDType]

type ImageLabelDataset = tf_Dataset[tuple[tf.Tensor, tf.Tensor]]


## Dataset

Using: https://www.kaggle.com/datasets/mantasu/glasses-detector


### Validate Dataset Path


In [ ]:
if not DS_PATH.exists():
	raise ValueError(f"Dataset path doesn't exist: {DS_PATH}")
if not DS_PATH.is_dir():
	raise ValueError(f"Dataset path is not a folder: {DS_PATH}")


### Dataset Directory Structure



```zsh
eza -TD
```
```zsh
 .
├──  cmu-face-images
│   ├──  test
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   ├──  train
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   └──  val
│       ├──  anyglasses
│       └──  no_anyglasses
├──  ex07
│   ├──  test
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   ├──  train
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   └──  val
│       ├──  anyglasses
│       └──  no_anyglasses
├──  face-attribute-2
│   ├──  train
│   │   └──  no_anyglasses
│   └──  val
│       └──  no_anyglasses
├──  face-attributes-extra
│   ├──  test
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   ├──  train
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   └──  val
│       ├──  anyglasses
│       └──  no_anyglasses
├──  face-attributes-grouped
│   ├──  test
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   ├──  train
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   └──  val
│       ├──  anyglasses
│       └──  no_anyglasses
├──  glasses-and-coverings
│   ├──  test
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   ├──  train
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   └──  val
│       ├──  anyglasses
│       └──  no_anyglasses
├──  glasses-detection
│   ├──  test
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   ├──  train
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   └──  val
│       ├──  anyglasses
│       └──  no_anyglasses
├──  glasses-image-dataset
│   ├──  test
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   ├──  train
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   └──  val
│       ├──  anyglasses
│       └──  no_anyglasses
├──  glasses-no-glasses
│   ├──  test
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   ├──  train
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   └──  val
│       ├──  anyglasses
│       └──  no_anyglasses
├──  indian-facial-database
│   ├──  test
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   ├──  train
│   │   ├──  anyglasses
│   │   └──  no_anyglasses
│   └──  val
│       ├──  anyglasses
│       └──  no_anyglasses
└──  sunglasses-no-sunglasses
    ├──  train
    │   ├──  anyglasses
    │   └──  no_anyglasses
    └──  val
        ├──  anyglasses
        └──  no_anyglasses
```


### Preparation and Serialisation

This should only need to be run once.

The dataset path contains subdatasets.

Each subdataset has a `train/` and `val/` folder, and an optional `test/` folder.

Within each of these folders is `anyglasses` and `no_anyglasses`, which contain the images and will be the labels. `face-attribute-2` does not contain `no_anyglasses`.


In [ ]:
def load_dataset(
	ds_path: Path, ignored: list[str], pattern: str = "*.jpg"
) -> dict[str, list[PathLabel]]:
	datasets: dict[str, list[PathLabel]] = {}
	for img_path in ds_path.rglob(pattern=pattern):
		subset: str
		split: str
		label: str
		subset, split, label = img_path.parts[-4:-1]
		if subset in ignored:
			continue
		if split not in datasets:
			datasets[split] = []
		datasets[split].append(
			(
				str(img_path.resolve()),
				bool(["no_anyglasses", "anyglasses"].index(label)),
			)
		)
	return datasets


raw_data: dict[str, list[PathLabel]] = load_dataset(
	ds_path=DS_PATH, ignored=IGNORE_SUBDATASETS
)
datasets: dict[str, npt.NDArray[np.str_]] = {}

for name, item in raw_data.items():
	datasets[name] = np.array(object=item, dtype=npd.StringDType())

	if VERBOSITY > 0:
		print(f"{name}: {len(item)}")
	if VERBOSITY > 1:
		display(
			[x for x in map(lambda i: ("/".join(i[0].split("/")[-4:]), i[1]), item)]
		)


#### Shuffle


In [ ]:
rng: np.random.Generator = np.random.default_rng(seed=SEED)
for i in datasets.values():
	rng.shuffle(x=i)

if VERBOSITY > 1:
	display(datasets)


### Data Ingestion Pipeline


In [ ]:
def build_dataset(
	data_array: npt.NDArray,
	/,
	shuffle: bool = False,
	repetitions: int = 0,
	caching: bool = True,
	use_disk: bool = True,
	*,
	verbosity: int = 0,
) -> ImageLabelDataset:
	"""Ingest pipeline."""

	paths: npt.NDArray = data_array[:, 0].astype(dtype=npd.StringDType)
	if verbosity > 2:
		display(paths)
	labels: npt.NDArray[LabelDType] = data_array[:, 1].astype(dtype="U5") == "True"
	labels = np.expand_dims(a=labels, axis=1)
	if verbosity > 1:
		display(labels)

	ds: ImageLabelDataset = tf_Dataset.from_tensor_slices(tensors=(paths, labels))

	def _load_image(
		filepath: tf.Tensor, label: tf.Tensor, /
	) -> tuple[tf.Tensor, tf.Tensor]:
		raw_image: tf.Tensor = tf.io.read_file(filepath)
		# Can also use tf.image.decode_png
		image: tf.Tensor = tf.io.decode_jpeg(raw_image, channels=3)
		##image = tf.image.resize(image, IMG_SIZE)
		##image.set_shape(IMG_SHAPE)
		return image, label

	# Order must be: `interleave -> batch -> map(time_consuming) -> cache -> shuffle -> repeat -> map(memory_consuming) -> batch -> prefetch`

	def _apply_stochastic_ops(
		ds: ImageLabelDataset, buffer_size: int, /
	) -> ImageLabelDataset:
		if shuffle:
			if verbosity > 0:
				print(f"Shuffling with buffer size {buffer_size}")
			ds = ds.shuffle(
				buffer_size=buffer_size, reshuffle_each_iteration=repetitions > 0
			)

		if repetitions == 0:
			return ds
		repeat_count: None | int = None if repetitions < 0 else repetitions
		if verbosity > 0:
			print(
				f"Repeating infinitely"
				if repeat_count is None
				else f"Repeating {repetitions} times"
			)
		return ds.repeat(count=repeat_count)

	def _build_scalable_ds(ds: ImageLabelDataset, /) -> ImageLabelDataset:
		"""### Sacrifice CPU Cycles (Per Epoch Decode Pipeline)

		`Order: shuffle(len(paths)) -> map -> batch (No cache)`
		"""

		if verbosity > 0:
			print("Not using cache")
		ds = _apply_stochastic_ops(ds, len(paths))
		ds = ds.map(map_func=_load_image, num_parallel_calls=tf_AUTOTUNE)
		return ds

	def _build_cache_ds(ds: ImageLabelDataset, use_disk: bool, /) -> ImageLabelDataset:
		"""### Sacrifice Perfect Randomness (Disk Cache Pipeline)

		`Order: map -> cache('/disk/path') -> shuffle(1000) -> batch`

		### Sacrifice System Memory (RAM Cache Pipeline)

		`Order: map -> cache() -> shuffle(len(paths)) -> batch`
		"""

		ds = ds.map(map_func=_load_image, num_parallel_calls=tf_AUTOTUNE)

		if use_disk:
			if verbosity > 0:
				print("Using disk cache")
			CACHE_PATH.parent.mkdir(
				parents=True, exist_ok=True
			)  # Make sure the path exists
			ds = ds.cache(filename=str(object=CACHE_PATH))
			ds = _apply_stochastic_ops(ds, BUFFER_SIZE)
		else:
			if verbosity > 0:
				print("Using RAM cache")
			ds = ds.cache()
			ds = _apply_stochastic_ops(ds, len(paths))
		return ds

	if caching:
		ds = _build_cache_ds(ds, use_disk)
	else:
		ds = _build_scalable_ds(ds)

	ds = ds.batch(batch_size=BATCH_SIZE)
	ds = ds.map(
		map_func=lambda image, label: (tf.cast(x=image, dtype=tf.uint8), label),
		num_parallel_calls=tf_AUTOTUNE,
	)

	ds = ds.prefetch(buffer_size=tf_AUTOTUNE)

	##tf.data.experimental.AutotuneOptions()
	##options = tf.data.Options()
	##options.autotune.enabled = True
	##options.autotune.ram_budget = 16642998272 // 10

	##ds = ds.with_options(options=options)

	return ds


In [ ]:
train_dataset: ImageLabelDataset = build_dataset(
	datasets["train"], shuffle=True, repetitions=TRAIN_REPETITIONS, verbosity=VERBOSITY
)
val_dataset: ImageLabelDataset = build_dataset(
	datasets["val"], repetitions=VAL_REPEITIONS
)
test_dataset: ImageLabelDataset = build_dataset(
	datasets["test"], repetitions=TEST_REPEITIONS
)


#### Inspect The Dataset


In [ ]:
def dataset_info(ds: ImageLabelDataset, /) -> None:
	print(f"Cardinality: {ds.cardinality()} ― Element spec: {ds.element_spec}")


if VERBOSITY > 0:
	dataset_info(train_dataset)
	dataset_info(val_dataset)
	dataset_info(test_dataset)
	##display(train_dataset.options().autotune.ram_budget)


In [ ]:
image_batch: tf.Tensor
label_batch: tf.Tensor
for image_batch, label_batch in train_dataset.take(count=1):
	for i in range(9):
		plt.subplot(3, 3, i + 1)
		plt.imshow(X=image_batch[i].numpy())
		plt.title(label=str(object=label_batch[i].numpy()[0]))
		plt.axis(False)


## Machine Learning


### Preparation

#### Calculate Values


In [ ]:
repetitions = (
	TRAIN_REPETITIONS if TRAIN_REPETITIONS >= 0 else TRAIN_REPETITIONS_IF_INFINITE
)
STEPS_PER_EPOCH = len(train_dataset) * (repetitions or 1) // BATCH_SIZE
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS

if VERBOSITY > 0:
	print(
		f"STEPS_PER_EPOCH = len(train_dataset) * repetitions // BATCH_SIZE:\n"
		f"\t{STEPS_PER_EPOCH} = {len(train_dataset)} * {repetitions or 1} // {BATCH_SIZE}\n"
	)
	print(
		f"TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS:\n\t{TOTAL_STEPS} = {STEPS_PER_EPOCH} * {MAX_EPOCHS}"
	)


#### Training Scheduler


In [ ]:
lr_schedule: k_optimisers.schedules.InverseTimeDecay = (
	k_optimisers.schedules.InverseTimeDecay(
		initial_learning_rate=INITIAL_LEARNING_RATE,
		decay_steps=STEPS_PER_EPOCH,
		decay_rate=DECAY_RATE,
	)
)


def get_optimiser() -> k_optimisers.Optimizer:
	return k_optimisers.Adam(learning_rate=lr_schedule)  # type: ignore


In [ ]:
# Demonstrate the scheduler
def fractional_power(x: int) -> int:
	EXP = 0.6
	SHIFT = 0
	SCALE = 9
	OFFSET = 8
	return int(SCALE * (MAX_EPOCHS + SHIFT) ** EXP - OFFSET)


count = fractional_power(MAX_EPOCHS)
if VERBOSITY > 0:
	print(count)

step: npt.NDArray[np.float64] = np.linspace(start=0, stop=TOTAL_STEPS, num=count)

lr = lr_schedule(step=step)

plt.figure(figsize=(8, 6))
plt.plot(step / STEPS_PER_EPOCH, lr)  # type: ignore
plt.xlim((0, MAX_EPOCHS))
plt.xlabel(xlabel="Epoch")
plt.ylabel(ylabel="Learning Rate")
plt.show()


#### Callbacks


In [ ]:
def get_callbacks() -> list[k.callbacks.Callback]:
	return [
		k.callbacks.EarlyStopping(
			monitor="val_loss", patience=STOPPING_PATIENCE, restore_best_weights=True
		)
	]


#### Plotting


In [ ]:
def get_plotter(metric: str) -> tfdocs_HistoryPlotter:
	return tfdocs_HistoryPlotter(metric=metric, smoothing_std=10)


history_dict: dict[str, Any] = {}


### Building The Model


#### Intake Layers


##### Inputs


In [ ]:
input_tensor = k.layers.Input(shape=IMG_SHAPE, dtype=ImageDType, name="InputLayer")


##### Preprocessing Pipeline Layer


In [ ]:
preprocess_pipeline: k.layers.Pipeline = k.layers.Pipeline(
	# Rescale the layers from 0-255 to 0-1
	layers=[
		k.layers.Resizing(
			height=IMG_SIZE[0],
			width=IMG_SIZE[1],
			pad_to_aspect_ratio=True,
			fill_value=0,
			name="resizing",
		),
		k.layers.Rescaling(scale=1.0 / 127.5, offset=-1, name="rescaling"),
	],
	name="preprocess_pipeline",
)

optional = preprocess_pipeline(input_tensor) if USE_PREPROCESSING else input_tensor


In [ ]:
image_batch: tf.Tensor
label_batch: tf.Tensor
for image_batch, label_batch in train_dataset.take(count=1):
	for i in range(9):
		plt.subplot(3, 3, i + 1)
		processed_image: ImageBatch = preprocess_pipeline(image_batch).numpy()
		plt.imshow(X=(processed_image[0] + 1) / 2)
		plt.axis(False)


##### Augmentation Pipeline Layer


In [ ]:
augmentation_pipeline: k.layers.Pipeline = k.layers.Pipeline(
	layers=[
		k.layers.RandomTranslation(
			height_factor=PREPROCESS_TRANSLATION_FACTOR,
			width_factor=PREPROCESS_TRANSLATION_FACTOR,
			fill_mode="constant",
			fill_value=0,
			name="rand_translation",
		),
		k.layers.RandomZoom(height_factor=PREPROCESS_ZOOM_FACTOR, name="rand_zoom"),
		k.layers.RandomRotation(
			factor=PREPROCESS_ROTATION_FACTOR,
			fill_mode="constant",
			fill_value=0,
			name="rand_rotation",
		),
		k_layers_RandomColourJitter(
			hue_factor=PREPROCESS_HUE_FACTOR, name="rand_colour_jitter"
		),
	],
	name="augmentation_pipeline",
)

input = augmentation_pipeline(optional) if USE_AUGMENTATIONS else optional


In [ ]:
def parse_image(image: tf.Tensor, /) -> tf.Tensor:
	return augmentation_pipeline(preprocess_pipeline(image))


mage_batch: tf.Tensor
label_batch: tf.Tensor
for image_batch, label_batch in train_dataset.take(count=1):
	for i in range(9):
		plt.subplot(3, 3, i + 1)
		augmented_image: ImageBatch = parse_image(image_batch).numpy()
		plt.imshow(X=augmented_image[0])
		plt.axis(False)


##### Normalisation Layer


In [ ]:
#input = k_layers_Normalisation(name="normalisation")(input)


#### Body


##### Stem


In [ ]:
stem = k.layers.Conv2D(
	filters=32,
	kernel_size=3,
	strides=2,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="stem_conv",
)(input)
stem = k_layers_BatchNormalisation(name="stem_bn")(stem)
stem = k.layers.LeakyReLU(name="stem_activation")(stem)


##### Block 1


In [ ]:
block1a_project = k.layers.Conv2D(
	filters=16,
	kernel_size=1,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block1a_project_conv",
)(stem)
block1a_project = k_layers_BatchNormalisation(name="block1a_project_bn")(
	block1a_project
)
#block1a_project = k.layers.LeakyReLU(name="block1a_project_activation")(block1a_project)


In [ ]:
block1b_project = k.layers.Conv2D(
	filters=16,
	kernel_size=3,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block1b_project_conv",
)(block1a_project)
block1b_project = k_layers_BatchNormalisation(name="block1b_project_bn")(
	block1b_project
)
#block1b_project = k.layers.LeakyReLU(name="block1b_project_activation")(block1b_project)


In [ ]:
block1_result = k.layers.Dropout(rate=DROPOUT_VAL, name="block1b_drop")(block1b_project)
block1_result = k.layers.Add(name="block1b_add")([block1_result, block1a_project])


##### Block 2


In [ ]:
block2a_expand = k.layers.Conv2D(
	filters=64,
	kernel_size=3,
	strides=2,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block2a_expand_conv",
)(block1_result)
block2a_expand = k_layers_BatchNormalisation(name="block2a_expand_bn")(block2a_expand)
block2a_expand = k.layers.LeakyReLU(name="block2a_expand_activation")(block2a_expand)

block2a_project = k.layers.Conv2D(
	filters=32,
	kernel_size=1,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block2a_project_conv",
)(block2a_expand)
block2a_project = k_layers_BatchNormalisation(name="block2a_project_bn")(
	block2a_project
)


In [ ]:
block2b_expand = k.layers.Conv2D(
	filters=128,
	kernel_size=3,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block2b_expand_conv",
)(block2a_project)
block2b_expand = k_layers_BatchNormalisation(name="block2b_expand_bn")(block2b_expand)
block2b_expand = k.layers.LeakyReLU(name="block2b_expand_activation")(block2b_expand)

block2b_project = k.layers.Conv2D(
	filters=32,
	kernel_size=3,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block2b_project_conv",
)(block2b_expand)
block2b_project = k_layers_BatchNormalisation(name="block2b_project_bn")(
	block2b_project
)


In [ ]:
block2_result = k.layers.Dropout(rate=DROPOUT_VAL, name="block2b_drop")(block2b_project)
block2_result = k.layers.Add(name="block2b_add")([block2_result, block2a_project])


##### Block 3


In [ ]:
block3a_expand = k.layers.Conv2D(
	filters=224,
	kernel_size=1,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block3a_expand_conv",
)(block2_result)
block3a_expand = k_layers_BatchNormalisation(name="block3a_expand_bn")(block3a_expand)
block3a_expand = k.layers.LeakyReLU(name="block3a_expand_activation")(block3a_expand)

block3a_depth = k.layers.DepthwiseConv2D(
	kernel_size=5,
	strides=2,
	padding="same",
	name="block3a_dwconv2",
)(block3a_expand)
block3a_depth = k_layers_BatchNormalisation(name="block3a_bn")(block3a_depth)
block3a_depth = k.layers.LeakyReLU(name="block3a_activation")(block3a_depth)

block3a_se = k.layers.GlobalAveragePooling2D(name="block3a_se_squeeze")(block3a_depth)
block3a_se = k.layers.Reshape(
	name="block3a_se_reshape", target_shape=(1, 1, 224)
)(block3a_se)
block3a_se = k.layers.Conv2D(
	filters=14,
	kernel_size=1,
	padding="same",
	activation=k.activations.relu,
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block3a_se_reduce",
)(block3a_se)
block3a_se = k.layers.Conv2D(
	filters=224,
	kernel_size=1,
	padding="same",
	activation=k.activations.sigmoid,
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block3a_se_expand",
)(block3a_se)
block3a_se = k.layers.Multiply(name="block3a_se_excite")([block3a_depth, block3a_se])

block3a_project = k.layers.Conv2D(
	filters=104,
	kernel_size=1,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block3a_project_conv",
)(block3a_se)
block3a_project = k_layers_BatchNormalisation(name="block3a_project_bn")(
	block3a_project
)


In [ ]:
block3b_expand = k.layers.Conv2D(
	filters=416,
	kernel_size=1,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block3b_expand_conv",
)(block3a_project)
block3b_expand = k_layers_BatchNormalisation(name="block3b_expand_bn")(block3b_expand)
block3b_expand = k.layers.LeakyReLU(name="block3b_expand_activation")(block3b_expand)

block3b_depth = k.layers.DepthwiseConv2D(
	kernel_size=5,
	padding="same",
	name="block3b_dwconv2",
)(block3b_expand)
block3b_depth = k_layers_BatchNormalisation(name="block3b_bn")(block3b_depth)
block3b_depth = k.layers.LeakyReLU(name="block3b_activation")(block3b_depth)

block3b_se = k.layers.GlobalAveragePooling2D(name="block3b_se_squeeze")(block3b_depth)
block3b_se = k.layers.Reshape(name="block3b_se_reshape", target_shape=(1, 1, 416))(
	block3b_se
)
block3b_se = k.layers.Conv2D(
	filters=26,
	kernel_size=1,
	padding="same",
	activation=k.activations.relu,
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block3b_se_reduce",
)(block3b_se)
block3b_se = k.layers.Conv2D(
	filters=416,
	kernel_size=1,
	padding="same",
	activation=k.activations.sigmoid,
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block3b_se_expand",
)(block3b_se)
block3b_se = k.layers.Multiply(name="block3b_se_excite")([block3b_depth, block3b_se])

block3b_project = k.layers.Conv2D(
	filters=104,
	kernel_size=1,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="block3b_project_conv",
)(block3b_se)
block3b_project = k_layers_BatchNormalisation(name="block3b_project_bn")(
	block3b_project
)


In [ ]:
block3_result = k.layers.Dropout(rate=DROPOUT_VAL, name="block3b_drop")(block3b_project)
block3_result = k.layers.Add(name="block3b_add")([block3_result, block3a_project])


##### Top


In [ ]:
top = k.layers.Conv2D(
	filters=352,
	kernel_size=1,
	padding="same",
	kernel_regularizer=k_regularisers.L2(l2=REGULARISER_VAL),
	name="top_conv",
)(block3_result)
top = k_layers_BatchNormalisation(name="top_bn")(top)
top = k.layers.LeakyReLU(name="top_activation")(top)


##### Feature Extraction


In [ ]:
output_tensor = k.layers.GlobalAveragePooling2D(name="global_average_pooling_2d")(top)
output_tensor = k.layers.Dropout(rate=0.3, name="dropout")(output_tensor)
output_tensor = k.layers.Dense(units=1, name="prediction")(output_tensor)


In [ ]:
model = k.Model(inputs=input_tensor, outputs=output_tensor, name="glasses_detector")
model.summary(line_length=100, positions=(0.3, 0.53, 0.64, 1.0), expand_nested=True)


### Executing The Model


#### Compile The Model


In [ ]:
model.compile(
	optimizer=get_optimiser(),
	loss=k.losses.BinaryCrossentropy(from_logits=True),
	metrics=[
		k.metrics.BinaryAccuracy()
		##k.metrics.BinaryCrossentropy()
	],
)


#### Training


In [ ]:
history_dict["model"] = model.fit(
	x=train_dataset,
	epochs=MAX_EPOCHS,
	verbose=str(object=VERBOSITY) if VERBOSITY_AFFECTS_FIT else "auto",
	callbacks=get_callbacks(),
	validation_data=val_dataset,
	##steps_per_epoch=STEPS_PER_EPOCH
)


### Model Evaluation


#### History Charts


In [ ]:
get_plotter(metric="loss").plot(histories=history_dict)


In [ ]:
get_plotter(metric="binary_accuracy").plot(histories=history_dict)


#### Evaluate Metrics


In [ ]:
loss: float
binary_accuracy: float
loss, binary_accuracy = model.evaluate(x=test_dataset)

print(f"Test binary_accuracy: {binary_accuracy}\nTest loss: {loss}")


#### Prediction Examples


In [ ]:
image_tensor: tf.Tensor
label_tensor: tf.Tensor
image_tensor, label_tensor = next(iter(test_dataset))
image_batch: ImageBatch = image_tensor.numpy()
label_batch: LabelBatch = label_tensor.numpy().flatten()
processed_images: tf.Tensor = preprocess_pipeline(image_tensor)
results_tensor: tf.Tensor = model(image_tensor, training=False)
results: LabelBatch = results_tensor.numpy().flatten()
predictions: npt.NDArray = results >= 0


if VERBOSITY > 0:
	print(f"Labels: {label_batch}")
	print(f"Predictions: {predictions}")

plt.figure(figsize=(10, 20))
for i in range(num := len(predictions) // 2):
	plt.subplot((num // 5) + 1, 5, i + 1)
	plt.imshow(X=(processed_images[i] + 1) / 2)
	plt.title(label=f"{label_batch[i]}:{predictions[i]}")
	plt.axis(False)


## Saving


In [ ]:
model.save(filepath=SAVE_PATH)
